In [ ]:
from pathlib import Path

import pandas as pd
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from itables import show
import requests

In [ ]:
addresses = pd.read_csv('data/output/geocoded_structured.csv')
addresses

In [ ]:
depots = pd.read_excel('data/raw/depots_geocoded.xlsx')
depots

In [ ]:
def insert_zero(row):
    row = str(row)
    if len(row) == 3:
        return '0' + row
    else:
        return row

addresses['Post Code (text)'] = addresses['Post Code (text)'].apply(insert_zero)
addresses

In [ ]:
# Make a mapping dictionary
lat_map = depots.set_index(depots['depot'].str.lower())['lat'].to_dict()
lng_map = depots.set_index(depots['depot'].str.lower())['lng'].to_dict()

# Map the lat/lng to the df based on charge_location
addresses['Longitude (depots)'] = addresses['Delivery Depot'].str.lower().map(lng_map)
addresses['Latitude (depots)'] = addresses['Delivery Depot'].str.lower().map(lat_map)

addresses


In [ ]:
addresses

validate the geocodes

In [ ]:
import unicodedata
import pandas as pd

def normalize_text(text):
    """Lowercase, strip spaces, and remove accents/macrons."""
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'  # remove diacritics/macrons
    )

def validate_geocodes(row):
    feature_name = normalize_text(row.get('Feature_Name'))
    suburb = normalize_text(row.get('Suburb'))
    place = normalize_text(row.get('Place'))
    
    if not feature_name:
        return False

    # Return True if there's a match or overlap with place or suburb
    return (
        feature_name == place or
        feature_name == suburb or
        feature_name in place or
        place in feature_name or
        feature_name in suburb or 
        suburb in feature_name
    )

In [ ]:
addresses_validated = addresses.copy()
addresses_validated['Validate'] = addresses.apply(validate_geocodes, axis=1)
addresses_validated[addresses_validated['Validate'] == False]

delete false validated and manual input

In [ ]:
print((144/4332) * 100)

In [ ]:
nz_full = ox.load_graphml('data/raw/new_zealand.graphml')


G = nz_full  # existing New Zealand road graph

# Step 1: Add nodes for ferry terminals
# Coordinates are approximate
north_terminal_id = 999000001
south_terminal_id = 999000002

G.add_node(north_terminal_id, x=174.7835723336724, y=-41.27874222366664)  # North Island terminal (Wellington)
G.add_node(south_terminal_id, x=174.00497, y=-41.28402)  # South Island terminal (Picton)

# Step 2: Add an edge representing the ferry
# 'length' can be distance in meters
ferry_distance = 53000  # approx 53 km

G.add_edge(north_terminal_id, south_terminal_id, length=ferry_distance, ferry=True)
G.add_edge(south_terminal_id, north_terminal_id, length=ferry_distance, ferry=True)

# Step 3: Plot the graph and highlight the ferry nodes
fig, ax = ox.plot_graph(
    G,
    node_size=10,
    edge_color='gray',
    edge_linewidth=0.5,
    show=False,
    close=False
)

# Highlight the ferry terminals
for node in [north_terminal_id, south_terminal_id]:
    x, y = G.nodes[node]['x'], G.nodes[node]['y']
    ax.scatter(x, y, c='blue', s=50)  # blue dot for ferry terminals

plt.show()

# Step 4: Example of using shortest_path across ferry
orig_node = north_terminal_id
dest_node = south_terminal_id
path = nx.shortest_path(G, orig_node, dest_node, weight='length')
print("Shortest path across ferry:", path)

In [ ]:
addresses_validated[addresses_validated['Longitude (mapbox)'].isna()]

In [ ]:
addresses_dropped_na = addresses_validated.copy()
addresses_dropped_na = addresses_dropped_na.drop(720)

In [ ]:
addresses_dropped_na[addresses_dropped_na['Longitude (mapbox)'].isna()]

In [ ]:
# Precompute nearest nodes
addresses_dropped_na['Orig_Node'] = ox.distance.nearest_nodes(G, X=addresses_dropped_na['Longitude (depots)'].values, Y=addresses_dropped_na['Latitude (depots)'].values)
addresses_dropped_na['Dest_Node'] = ox.distance.nearest_nodes(G, X=addresses_dropped_na['Longitude (mapbox)'].values, Y=addresses_dropped_na['Latitude (mapbox)'].values)

addresses_dropped_na

In [ ]:
# Compute distances without GDF overhead
def fast_distance(row):
    try:
        distance = nx.shortest_path_length(G, row['Orig_Node'], row['Dest_Node'], weight="length") / 1000
        return round(distance, 2)
    except nx.NetworkXNoPath:
        return None

addresses_dropped_na['Distance (mapbox)'] = addresses_dropped_na.apply(fast_distance, axis=1)

In [ ]:
Path('data/output').mkdir(parents=True, exist_ok=True)
addresses_dropped_na.to_csv('data/output/distances_mapbox_google.csv', index=False)

In [ ]:
addresses_dropped_na[addresses_dropped_na['Distance (mapbox)'].isna()]

In [ ]:
def marker(row):
    manual = row['Manual_Input']
    validate = row['Validate']
    empty_distance = row['Distance (mapbox)']

    if manual == 'y' or validate == False or pd.isna(empty_distance):
        return True
    return False

In [ ]:
final_df = addresses_dropped_na.copy()
final_df['Final Marker'] = final_df.apply(marker, axis=1)
final_df

In [ ]:
final_df[final_df['Final Marker'] == True]

In [ ]:
Path('data/output').mkdir(parents=True, exist_ok=True)
final_df.to_excel('data/output/for_manual_input.xlsx')